# Adaptive Low-Latency Fraud Detection in Streaming Financial Systems with LLM-Augmented Explainability

## Notebook 02 — Streaming Preprocessing & Leakage-Safe Transformation

**M.Tech Thesis**  

### Research Context & Purpose

In streaming financial fraud detection, data leakage is a critical threat to scientific validity. Standard offline preprocessing techniques—such as global mean imputation, global target encoding, or full-dataset standardization—leak future information into past observations, yielding deceptively optimistic performance estimates that collapse in production.

This notebook establishes, validates, and exercises the **leakage-safe streaming preprocessing infrastructure** implemented in `src/data/preprocessing.py` and `src/data/loader.py`.

### Methodological Scope
- **Warmup-Only State Estimation:** All imputation statistics, categorical index encodings, and scaling parameters are fitted exclusively on the historical warmup window ($t_0 \rightarrow t_{\text{warmup}}$).
- **Prequential Sample Transformation:** Streaming transactions are transformed sample-by-sample or batch-by-batch without updating fitted parameters from test-stream samples.
- **Strict Isolation:** Identifiers (`TransactionID`), temporal anchors (`TransactionDT`), labels (`isFraud`), and categorical segments (`ProductCD`) are strictly partitioned from the learner feature matrix $X_t$.
- **Robust Unseen Category Handling:** Novel categorical values encountered in production are cleanly mapped to an explicit unknown index ($0$), preventing pipeline crashes.
- **Dual Execution Modes:** Provides both a fast, lightweight local smoke test (`LOCAL_CPU`) and a complete full-dataset execution pipeline (`KAGGLE`).


## 1. Import Required Libraries and Environment Setup

We import standard numerical, data manipulation, and plotting libraries. All thesis-quality plotting conventions and random seeds are initialized here.


In [ ]:
from pathlib import Path
import sys
import os
import json
import time
import shutil
import zipfile
import warnings
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# Thesis visual presentation standards
plt.style.use("default")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 10
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 11

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", None)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Environment configured successfully. Random seed locked to 42.")


## 2. Execution Mode Configuration

To support both rapid local engineering validation and full-scale empirical processing on Kaggle, this notebook provides a single standardized top-level configuration toggle.

- **`LOCAL_CPU` (Smoke Test):** Operates on a lightweight slice (e.g., 5,000 rows) using local paths. Validates imports, classes, functions, schema integrity, and diagnostic plots with zero runtime errors.
- **`KAGGLE` (Full Execution):** Operates on the full benchmark dataset inside Kaggle. Automatically configures paths, executes the complete prequential pipeline, and generates exportable artifact bundles.


In [ ]:
# TOP-LEVEL EXECUTION MODE SELECTION

# Options:
#   - "LOCAL_CPU" : Fast smoke test on small row slice (verifies code, imports, logic)
#   - "KAGGLE"    : Full execution on complete benchmark dataset in Kaggle environment

RUN_MODE = "LOCAL_CPU"  # Change to "KAGGLE" when executing in Kaggle environment

# Runtime parameters conditioned on execution mode
if RUN_MODE == "LOCAL_CPU":
    SMOKE_TEST_ROWS = 5000
    WARMUP_RATIO = 0.20
    VERBOSE = True
    print("[RUN_MODE] Running in LOCAL_CPU mode (Lightweight smoke test, 5,000 rows).")
elif RUN_MODE == "KAGGLE":
    SMOKE_TEST_ROWS = None  # None indicates full dataset loading
    WARMUP_RATIO = 0.20
    VERBOSE = True
    print("[RUN_MODE] Running in KAGGLE mode (Full benchmark execution).")
else:
    raise ValueError(f"Unknown RUN_MODE: '{RUN_MODE}'. Expected 'LOCAL_CPU' or 'KAGGLE'.")


## 3. Repository Auto-Discovery & Path Configuration

The notebook dynamically detects its execution environment. When running locally, it locates `pyproject.toml` to anchor the repository root. When running on Kaggle, it handles repository cloning and adds `src/` to the Python module search path.


In [ ]:
def find_project_root(start_path: Path) -> Path:
    """Locate repository root by traversing parent directories for pyproject.toml."""
    current = start_path.resolve()
    while current != current.parent:
        if (current / "pyproject.toml").exists():
            return current
        current = current.parent
    return None

if RUN_MODE == "LOCAL_CPU":
    PROJECT_ROOT = find_project_root(Path.cwd())
    if PROJECT_ROOT is None:
        PROJECT_ROOT = Path.cwd().resolve()
        print(f"Warning: pyproject.toml not found in parents. Assuming {PROJECT_ROOT}")
    else:
        print(f"Project root identified: {PROJECT_ROOT}")

elif RUN_MODE == "KAGGLE":
    # On Kaggle, repository is cloned into /kaggle/working if not already present
    KAGGLE_WORKING = Path("/kaggle/working")
    REPO_DIR = KAGGLE_WORKING / "adaptive-low-latency-streaming-financial-fraud-detection-with-llm-explainability"
    
    if not REPO_DIR.exists():
        print("Cloning repository into Kaggle working environment...")
        clone_cmd = (
            "git clone https://github.com/ParminderSinghGithub/"
            "adaptive-low-latency-streaming-financial-fraud-detection-with-llm-explainability.git"
        )
        os.system(f"cd {KAGGLE_WORKING} && {clone_cmd}")
        print("Repository clone complete.")
    
    PROJECT_ROOT = REPO_DIR

# Ensure repository root is in sys.path for clean src imports
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
    print(f"Appended to sys.path: {PROJECT_ROOT}")

# Standardized output directories
OUTPUT_DIR = PROJECT_ROOT / "outputs"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
KAGGLE_ARTIFACT_DIR = PROJECT_ROOT / "kaggle_artifacts"

for d in [TABLE_DIR, FIGURE_DIR, CHECKPOINT_DIR, KAGGLE_ARTIFACT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Output directories verified under: {OUTPUT_DIR}")


## 4. Output & Figure Export Utilities

We define standard helper functions to export figures (`.png`, 300 DPI) and tables (`.csv`) to `outputs/` without polluting the workspace.


In [ ]:
def save_table(df: pd.DataFrame, filename: str) -> Path:
    """Save dataframe as CSV into outputs/tables."""
    path = TABLE_DIR / f"{filename}.csv"
    df.to_csv(path, index=False)
    print(f"Table saved: {path.name} ({len(df)} rows)")
    return path

def save_figure(fig, filename: str) -> Path:
    """Save matplotlib figure with thesis-quality DPI."""
    path = FIGURE_DIR / f"{filename}.png"
    fig.savefig(path, dpi=300, bbox_inches="tight")
    print(f"Figure saved: {path.name}")
    return path


## 5. Dataset Configuration & Dynamic Path Resolution

The notebook automatically locates the dataset directory regardless of environment:
- In `LOCAL_CPU` mode: resolves `datasets/` under the repository root.
- In `KAGGLE` mode: searches `/kaggle/input/` dynamically for `train_transaction.csv` across all dataset subfolders or automatically extracts dataset `.zip` archives if present.


In [ ]:
def resolve_ieee_cis_dir() -> Path:
    """Dynamically locate the directory containing train_transaction.csv."""
    if RUN_MODE == "LOCAL_CPU":
        local_data = PROJECT_ROOT / "datasets"
        direct = local_data / "ieee_cis"
        if (direct / "train_transaction.csv").exists():
            return direct
        matches = list(local_data.rglob("train_transaction.csv"))
        if matches:
            return matches[0].parent
        raise FileNotFoundError(f"train_transaction.csv not found under local datasets: {local_data}")
    
    elif RUN_MODE == "KAGGLE":
        kaggle_input = Path("/kaggle/input")
        
        # 1. Search recursively for train_transaction.csv in /kaggle/input
        if kaggle_input.exists():
            matches = list(kaggle_input.rglob("train_transaction.csv"))
            if matches:
                print(f"Located train_transaction.csv in Kaggle input: {matches[0]}")
                return matches[0].parent
        
        # 2. Check if a zip archive is present in /kaggle/input and extract it
        if kaggle_input.exists():
            zips = list(kaggle_input.rglob("*.zip"))
            if zips:
                extract_dir = Path("/kaggle/working/datasets")
                extract_dir.mkdir(parents=True, exist_ok=True)
                for z in zips:
                    print(f"Extracting {z.name} to {extract_dir}...")
                    with zipfile.ZipFile(z, "r") as zip_ref:
                        zip_ref.extractall(extract_dir)
                matches = list(extract_dir.rglob("train_transaction.csv"))
                if matches:
                    print(f"Located extracted train_transaction.csv: {matches[0]}")
                    return matches[0].parent
        
        # 3. Check fallback repository directory
        fallback = PROJECT_ROOT / "datasets"
        matches = list(fallback.rglob("train_transaction.csv"))
        if matches:
            return matches[0].parent
        
        raise FileNotFoundError(
            "Could not locate train_transaction.csv anywhere under /kaggle/input. "
            "Ensure the IEEE-CIS dataset or zip archive is added to the Kaggle notebook input."
        )

IEEE_CIS_DIR = resolve_ieee_cis_dir()
DATASET_ROOT = IEEE_CIS_DIR.parent
print(f"IEEE-CIS data directory resolved to: {IEEE_CIS_DIR}")


## 6. Environment & Dependency Verification

We verify that the core runtime dependencies are installed and compatible.


In [ ]:
dependency_manifest = [
    ("Python", sys.version.split()[0]),
    ("NumPy", np.__version__),
    ("Pandas", pd.__version__),
    ("Matplotlib", plt.matplotlib.__version__),
]

dep_df = pd.DataFrame(dependency_manifest, columns=["Dependency", "Installed Version"])
display(dep_df)

# Confirm src modules can be cleanly imported
from src.data.loader import load_ieee_cis, get_dataset_metadata
from src.data.preprocessing import StreamingPreprocessor

print("Repository modules (src.data.loader, src.data.preprocessing) imported cleanly.")


## 7. Dataset Ingestion (IEEE-CIS Benchmark)

We load the IEEE-CIS fraud detection dataset using the authoritative repository loader `load_ieee_cis()`.
- In `LOCAL_CPU` mode, we load a controlled slice (5,000 samples) to verify end-to-end functionality.
- In `KAGGLE` mode, the full dataset (590,540 rows) is loaded for production-scale preprocessing.


In [ ]:
print(f"Loading IEEE-CIS transaction data (RUN_MODE={RUN_MODE})...")
t_start = time.time()

# If in LOCAL_CPU mode, read only a slice of rows for fast smoke verification
if RUN_MODE == "LOCAL_CPU":
    tx_file = IEEE_CIS_DIR / "train_transaction.csv"
    id_file = IEEE_CIS_DIR / "train_identity.csv"
    
    df_tx = pd.read_csv(tx_file, nrows=SMOKE_TEST_ROWS)
    if id_file.exists():
        df_id = pd.read_csv(id_file)
        df_raw = pd.merge(df_tx, df_id, on="TransactionID", how="left")
    else:
        df_raw = df_tx
else:
    # Full dataset load via repository module
    df_raw = load_ieee_cis(data_dir=IEEE_CIS_DIR, split="train", join_identity=True)

load_duration = time.time() - t_start
print(f"Ingestion complete in {load_duration:.2f}s. Loaded shape: {df_raw.shape}")


## 8. Stream Schema & Temporal Order Inspection

Streaming financial systems require strict chronological ordering by transaction time ($t$).
Here, we inspect the temporal anchor `TransactionDT` and verify that transactions are properly sorted.


In [ ]:
# Sort strictly by temporal timestamp to guarantee stream replay integrity
df_raw = df_raw.sort_values("TransactionDT").reset_index(drop=True)

# Verification assertions
assert "TransactionDT" in df_raw.columns, "TransactionDT missing from schema."
assert "isFraud" in df_raw.columns, "isFraud target column missing."
assert "ProductCD" in df_raw.columns, "ProductCD segment column missing."

# Verify temporal monotonicity
is_sorted = df_raw["TransactionDT"].is_monotonic_increasing
print(f"Temporal order strictly non-decreasing: {is_sorted}")
assert is_sorted, "Temporal sorting violated."

stream_stats = pd.DataFrame([
    {"Metric": "Total Transactions Loaded", "Value": f"{len(df_raw):,}"},
    {"Metric": "Total Raw Features", "Value": f"{df_raw.shape[1]}"},
    {"Metric": "Min Timestamp (TransactionDT)", "Value": f"{df_raw['TransactionDT'].min():,}"},
    {"Metric": "Max Timestamp (TransactionDT)", "Value": f"{df_raw['TransactionDT'].max():,}"},
    {"Metric": "Fraud Rate (%)", "Value": f"{df_raw['isFraud'].mean() * 100:.3f}%"},
    {"Metric": "Categorical Segments (ProductCD)", "Value": str(sorted(df_raw['ProductCD'].dropna().unique().tolist()))}
])
display(stream_stats)
save_table(stream_stats, "stream_schema_inspection")


## 9. Initialize Leakage-Safe Preprocessor

We instantiate the authoritative `StreamingPreprocessor` from `src.data.preprocessing`.

- **`scale_features=False`:** Feature scaling is disabled by default for tree-based streaming models (e.g., Hoeffding Trees) to avoid distortion of raw thresholds and reduce runtime overhead.
- **Segment Column:** Automatically assigned to `ProductCD` for IEEE-CIS, enabling segment-aware drift isolation ($P3$).


In [ ]:
preprocessor = StreamingPreprocessor(
    dataset_name="ieee_cis",
    scale_features=False,
    custom_categorical_cols=["ProductCD", "card4", "card6", "P_emaildomain", "R_emaildomain"]
)

print("StreamingPreprocessor initialized.")
print(f"Target column isolated: {preprocessor.target_col}")
print(f"Temporal column isolated: {preprocessor.temporal_col}")
print(f"Identifier column isolated: {preprocessor.id_col}")
print(f"Segment column designated: {preprocessor.segment_col}")


## 10. Historical Warmup Fitting ($t_0 \rightarrow t_{\text{warmup}}$)

To prevent lookahead and data leakage, all preprocessing statistics are estimated strictly from historical warmup transactions.

The dataset is partitioned into:
1. **Warmup Historical Window ($W_{\text{warmup}}$):** Initial 20% of chronological transactions.
2. **Streaming Evaluation Window ($W_{\text{stream}}$):** Remaining 80% of transactions.


In [ ]:
n_total = len(df_raw)
n_warmup = int(n_total * WARMUP_RATIO)

df_warmup = df_raw.iloc[:n_warmup].copy()
df_stream = df_raw.iloc[n_warmup:].copy()

print(f"Warmup window size:   {len(df_warmup):,} transactions (first {WARMUP_RATIO*100:.0f}%)")
print(f"Streaming window size: {len(df_stream):,} transactions (subsequent {(1-WARMUP_RATIO)*100:.0f}%)")

# Execute warmup fitting
t_fit_start = time.time()
preprocessor.fit(df_warmup)
fit_duration = time.time() - t_fit_start

print(f"Warmup fitting executed in {fit_duration:.3f} seconds.")
assert preprocessor.is_fitted, "Preprocessor failed to set is_fitted=True."

warmup_fit_summary = pd.DataFrame([
    {"Property": "Numerical Features Identified", "Count": len(preprocessor.numerical_cols)},
    {"Property": "Categorical Features Identified", "Count": len(preprocessor.categorical_cols)},
    {"Property": "Total Transformed Features", "Count": len(preprocessor.feature_names)},
    {"Property": "Imputation Medians Fitted", "Count": len(preprocessor.imputation_medians)},
    {"Property": "Category Encodings Built", "Count": len(preprocessor.category_mappings)}
])
display(warmup_fit_summary)
save_table(warmup_fit_summary, "warmup_fit_summary")


## 11. Streaming Transformation Execution

We now execute the streaming transformation on the out-of-sample evaluation stream ($df_{\text{stream}}$).
The preprocessor:
- Imputes missing numerical values using pre-computed warmup medians.
- Maps categorical features to index integers based on warmup vocabularies.
- Isolates and returns feature matrix $X$, ground-truth label series $y$, and segment series $seg$.


In [ ]:
t_trans_start = time.time()
X_stream, y_stream, seg_stream = preprocessor.transform(df_stream)
trans_duration = time.time() - t_trans_start

throughput = len(df_stream) / trans_duration if trans_duration > 0 else 0
print(f"Streaming transformation executed in {trans_duration:.3f}s ({throughput:,.0f} samples/sec).")
print(f"Transformed Feature Matrix X: {X_stream.shape}")
print(f"Transformed Labels y:          {y_stream.shape}")
print(f"Segment Series seg:           {seg_stream.shape if seg_stream is not None else 'None'}")

# Validate shapes
assert len(X_stream) == len(df_stream), "Row count mismatch in transformed features."
assert len(y_stream) == len(df_stream), "Row count mismatch in transformed labels."
assert len(seg_stream) == len(df_stream), "Row count mismatch in segment series."

sample_preview = X_stream.iloc[:5, :8]
display(sample_preview)


## 12. Numerical Missing-Value Imputation Validation

We verify that:
1. No missing values (`NaN`) remain in any numerical feature within $X_{\text{stream}}$.
2. The values used for imputation strictly match the historical warmup medians.


In [ ]:
# Check missing value count across all numerical features
num_cols = preprocessor.numerical_cols
nan_counts = X_stream[num_cols].isna().sum().sum()

print(f"Total NaN values remaining in numerical features: {nan_counts}")
assert nan_counts == 0, "Numerical missing value imputation incomplete."

# Verify that imputed values match warmup medians on a sample feature
sample_num_feat = [c for c in num_cols if c in preprocessor.imputation_medians][0]
warmup_median = preprocessor.imputation_medians[sample_num_feat]

print(f"Sample Feature: '{sample_num_feat}' | Historical Warmup Median: {warmup_median}")
assert not np.isnan(warmup_median), "Warmup median is NaN."
print("Numerical imputation validation: PASSED.")


## 13. Categorical Feature Encoding & Unseen Category Handling

In production financial streams, previously unseen categorical values frequently appear.
The preprocessor must:
- Map known warmup categories to positive integer indices ($1, 2, \dots$).
- Map novel/unseen categories and missing values to index `0` ([UNKNOWN]) without throwing runtime errors.


In [ ]:
sample_cat_col = preprocessor.categorical_cols[0]
mapping = preprocessor.category_mappings[sample_cat_col]

print(f"Sample Categorical Feature: '{sample_cat_col}'")
print(f"Known Warmup Categories ({len(mapping)}): {mapping}")

# Test handling of an unseen category via single-sample DataFrame transformation
test_sample_df = df_stream.iloc[:1].copy()
test_sample_df[sample_cat_col] = "UNSEEN_NOVEL_CATEGORY_9999"

x_single, _, _ = preprocessor.transform(test_sample_df)

transformed_val = x_single[sample_cat_col].iloc[0]
print(f"Transformed value for unseen category: {transformed_val}")
# Preprocessor maps unseen categories and NaNs to 0 [UNKNOWN]
assert transformed_val == 0, f"Unseen category was not mapped to index 0: got {transformed_val}"
print("Unseen category handling validation: PASSED.")


## 14. Feature, Target, Temporal, and Identifier Isolation Check

To prevent identity memorization and direct label leakage, non-feature columns must be strictly excluded from the transformed feature matrix $X$.


In [ ]:
excluded_candidates = ["TransactionID", "TransactionDT", "isFraud"]

for col in excluded_candidates:
    is_present = col in X_stream.columns
    print(f"Checking exclusion of '{col}': Present in X? -> {is_present}")
    assert not is_present, f"CRITICAL LEAKAGE: '{col}' found in feature matrix X!"

# Confirm target series contains valid binary labels
unique_labels = sorted(y_stream.unique().tolist())
print(f"Isolated Target Labels: {unique_labels}")
assert set(unique_labels).issubset({0, 1}), f"Invalid target labels: {unique_labels}"

# Confirm segment series is populated
assert seg_stream is not None, "Segment series is None."
print(f"Segment Series non-null count: {seg_stream.notna().sum():,} / {len(seg_stream):,}")
print("Column isolation checks: PASSED.")


## 15. Anti-Leakage Safeguard Verification

We execute rigorous programmatic checks confirming that:
1. Preprocessor state is completely frozen after warmup fitting.
2. The streaming transformation did not alter any warmup medians or mappings.
3. No information from the streaming window flowed backward into fitted state.


In [ ]:
# Snapshot preprocessor state before and after streaming
medians_snapshot = preprocessor.imputation_medians.copy()
mappings_snapshot = {k: v.copy() for k, v in preprocessor.category_mappings.items()}

# Execute another streaming batch transformation
_ = preprocessor.transform(df_stream.iloc[:100])

# Assert frozen state integrity
assert preprocessor.imputation_medians == medians_snapshot, "Imputation medians mutated during streaming!"
assert preprocessor.category_mappings == mappings_snapshot, "Category mappings mutated during streaming!"

print("Anti-leakage safeguard verification assertions: ALL PASSED.")


## 16. Preprocessed Stream Diagnostics & Visualizations

We generate diagnostic visual summaries of feature types and transaction distribution across segments.


In [ ]:
# Feature type distribution plot
feature_type_counts = pd.Series({
    "Numerical Features": len(preprocessor.numerical_cols),
    "Categorical Features": len(preprocessor.categorical_cols)
})

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(feature_type_counts.index, feature_type_counts.values, color=["#1f77b4", "#ff7f0e"])
ax.set_title("Preprocessed Feature Type Breakdown (IEEE-CIS)")
ax.set_ylabel("Number of Features")
ax.bar_label(bars, padding=3)
plt.tight_layout()
save_figure(fig, "preprocessed_feature_types")
plt.show()

# Segment distribution plot
fig, ax = plt.subplots(figsize=(8, 4))
seg_counts = seg_stream.value_counts()
bars = ax.bar(seg_counts.index.astype(str), seg_counts.values, color="#2ca02c")
ax.set_title("Streaming Transaction Count by Categorical Segment (ProductCD)")
ax.set_xlabel("Segment (ProductCD)")
ax.set_ylabel("Transactions")
ax.bar_label(bars, padding=3, fmt="%.0f")
plt.tight_layout()
save_figure(fig, "streaming_segment_distribution")
plt.show()


## 17. Preprocessor State Serialization & Bitwise Reproducibility

For streaming pipelines and reproducible benchmarks, the preprocessor state must be serializable to JSON and reconstructible with bitwise-identical output.


In [ ]:
checkpoint_path = CHECKPOINT_DIR / "streaming_preprocessor_ieee_cis.json"

preprocessor.save_state(checkpoint_path)
print(f"Serialized preprocessor state exported to: {checkpoint_path.name}")

# Reconstruct from serialized state
reconstructed_preprocessor = StreamingPreprocessor.load_state(checkpoint_path)

# Verify bitwise identical transformation on test slice
X_orig, _, _ = preprocessor.transform(df_stream.iloc[:50])
X_recon, _, _ = reconstructed_preprocessor.transform(df_stream.iloc[:50])

pd.testing.assert_frame_equal(X_orig, X_recon)
print("Bitwise reproducibility validation: PASSED.")


## 18. Kaggle Execution Notes & Artifact Packaging

When executing on Kaggle in `KAGGLE` mode, this cell packages all generated tables, figures, and serialized state into a single ZIP archive for simple one-click download.


In [ ]:
def package_run_artifacts() -> Path:
    """Bundle outputs into a timestamped zip archive for download."""
    archive_path = KAGGLE_ARTIFACT_DIR / "notebook02_preprocessing_artifacts.zip"
    
    with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as zip_file:
        for folder in [TABLE_DIR, FIGURE_DIR, CHECKPOINT_DIR]:
            for file_path in folder.glob("*.*"):
                arcname = f"{folder.name}/{file_path.name}"
                zip_file.write(file_path, arcname=arcname)
    
    size_kb = archive_path.stat().st_size / 1024
    print(f"Artifact archive created: {archive_path.name} ({size_kb:.1f} KB)")
    return archive_path

artifact_zip = package_run_artifacts()
print(f"Artifact archive ready at: {artifact_zip}")


## 19. Methodological Execution Checklist

This checklist confirms that all required engineering and scientific constraints for streaming preprocessing have been satisfied during execution.

| Check | Requirement | Verification Method | Status |
| :--- | :--- | :--- | :--- |
| **C1** | Zero Future-Data Leakage | Warmup-only statistic estimation ($t_0 \rightarrow t_{\text{warmup}}$) | Validated |
| **C2** | Zero Target Leakage | Explicit exclusion of `isFraud` from feature matrix $X$ | Validated |
| **C3** | Temporal Ordering | Strict sorting by `TransactionDT` preserved | Validated |
| **C4** | Robust Unseen Categories | Novel values mapped to index `0` without errors | Validated |
| **C5** | Segment Feature Preservation | Categorical segment column (`ProductCD`) cleanly isolated | Validated |
| **C6** | Bitwise State Reproducibility | Serialized JSON roundtrip yields identical transformations | Validated |
| **C7** | Dual Execution Modes | `LOCAL_CPU` smoke test and `KAGGLE` full execution supported | Configured |

*(Note: Empirical fraud detection performance metrics, PR-AUC curves, and latency benchmarks are evaluated in subsequent experimental notebooks E1–E10).*
